In [ ]:
from torch import nn, permute
import torch
import torch.nn.functional as F
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

IMG_H=np.short(128)
IMG_W=np.short(256)
IMG_C=3
NET_NAME='VAENet'

In [ ]:
from pytorch_nndct.apis import Inspector

In [ ]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out


In [ ]:
# Specify a target name or fingerprint you want to deploy on
target = "DPUCZDX8G_ISA1_B4096"
# Initialize inspector with target
inspector = Inspector(target)

In [ ]:
# Start to inspect the float model
# Note: visualization of inspection results relies on the dot engine.If you don't install dot successfully, set 'image_format = None' when inspecting.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = vaemodel1()
model.load_state_dict(torch.load('pre_trained_w_encoder.pt'), strict=True)

In [ ]:
dummy_input = torch.randn(1, IMG_C, IMG_H, IMG_W)
os.makedirs(NET_NAME, exist_ok=True)
inspector.inspect(model, (dummy_input,), device=device, output_dir=f"{NET_NAME}/inspect", image_format="png") 

In [ ]:
# Show the dot image
from IPython.display import Image
Image(f"{NET_NAME}/inspect/inspect_{target}.png")

In [ ]:
from pytorch_nndct.apis import torch_quantizer
from torch.utils.data import DataLoader, Dataset
import pathlib
import typing

def quantize_model(model: nn.Module, quant_mode: str, device: torch.device, quant_stub: typing.List[torch.Tensor],
                   quant_dataset: Dataset, output_path: pathlib.Path, batch_size: int,
                   num_workers: int = 4):
    """
    Configures quantization functions and invokes evaluation to generate quantized model.
    Args:
        model (torch.nn.Module):  Float model to quantize.
        quant_mode (str): Quantizer mode.  Either 'calib' or 'test'.
        device (torch.device): Device to use for model quantization.
        quant_stub (torch.Tensor): Random tensor with same dimensionality as real input.
        quant_dataset (torch.utils.data.Dataset):  Dataset of samples used for quantization.
        output_path (pathlib.Path): Output path for quantization artifacts.
        batch_size (int): Batch size for quantization.
        num_workers (int): Number of workers for the DataLoader.
    Returns:
        None for calibration mode, quantized inference model for test mode.
    """
    if quant_mode == 'calib':
        actual_batch_size = batch_size
    elif quant_mode == 'test':
        actual_batch_size = 1
    else:
        raise ValueError('ERROR: Unknown quant_mode {}!'.format(quant_mode))
    quant_loader = DataLoader(quant_dataset, batch_size=actual_batch_size, shuffle=True, num_workers=num_workers,
                                  pin_memory=True)
    quantizer = torch_quantizer(quant_mode, model, quant_stub, output_dir=str(output_path), device=device)
    quant_model = quantizer.quant_model
    evaluate_model(quant_model, quant_loader, device, quantizer, quant_mode, output_path)
    if quant_mode == 'calib':
        return None
    elif quant_mode == 'test':
        return quant_model

def evaluate_model(model: nn.Module, dataloader: DataLoader, device: torch.device, quantizer: torch_quantizer,
                   quant_mode: str, output_path: pathlib.Path):
    """
    Evaluates a quantizable model on the specified device according to the supplied training split subset.
    Required to calibrate the quantization output artifacts.
    Args:
        model (tnn.Module): Model to quantize.
        dataloader (torch.utils.data.DataLoader): Dataloader with quantization subset of training split.
        device (torch.device): Compute device to quantize on.
        quantizer (pytorch_nndct.apis.torch_quantizer): Xilinx quantization module.
        quant_mode (str): Quantization mode to run in.  Either 'calib' or 'test'.
        output_path (pathlib.Path): Output path for quantization artifacts.
    """
    model.eval()
    with torch.no_grad():
        for in_tensor, _ in dataloader:
            in_tensor = in_tensor.to(device)
            out_tensor = model(in_tensor)
        if quant_mode == 'test':
            print('Network output shape = {}'.format(out_tensor[0].shape))

    if quant_mode == 'calib':
        quantizer.export_quant_config()
    if quant_mode == 'test':
        quantizer.export_xmodel(deploy_check=True, output_dir=str(output_path))

In [ ]:
#-----------------------------------------------------------------------------#
#    IMAGE PREPROCESSING                                                      #
#-----------------------------------------------------------------------------#
## DIRECTORIES with datasets
img_path = 'dataset'

# Dataset with Transformation (using only 100 images)
full_dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.0, 0.0, 0.0], 
            std=[1.0, 1.0, 1.0])
    ])
)

# Data split
trainpctg  = 0.8
train_size = int(trainpctg * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, test_size])

print(f"Training set size: {len(train_set)}, Testing set size: {len(val_set)}")

In [ ]:
quant_dir = "./VAENet/"
model = vaemodel1().to(device)

# Quantization
quantize_model(model, 'calib', device, dummy_input, train_set, quant_dir, 1)
quantize_model(model, 'test', device, dummy_input, val_set, quant_dir, 1)

In [ ]:
!vai_c_xir --xmodel ./VAENet/vaemodel1_int.xmodel --arch /opt/vitis_ai/compiler/arch/DPUCZDX8G/ZCU104/arch.json --output_dir ./VAENet --net_name zcu104_vaemodel1